In [ ]:
# config bootstrap (auto-added): resolve repo paths from config.py
import os as _os, sys as _sys
_h = _os.path.abspath(_os.getcwd())
while not _os.path.exists(_os.path.join(_h, 'config.py')) and _os.path.dirname(_h) != _h:
    _h = _os.path.dirname(_h)
_sys.path.insert(0, _h)
import config as _cfg

# Text Fact-Checking Module — DeBERTa-v3 NLI + TF-IDF Sentence Extraction

Uses `cross-encoder/nli-deberta-v3-large` to check if an article's text **entails** or
**contradicts** its caption. Instead of feeding the full article (which gets truncated),
we first extract the **3 most relevant sentences** using TF-IDF cosine similarity against
the caption, then use those as the NLI premise.

In [1]:
import json
import os
import re

import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

d:\Pics Can Lie\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch: 2.6.0+cu124
CUDA: True
GPU: NVIDIA GeForce RTX 4060 Laptop GPU


In [2]:
# ── Config ──
DATASET_ROOT     = _os.path.join(str(_cfg.ROOT), 'datasets', 'dataset')
ANNOTATIONS_PATH = os.path.join(DATASET_ROOT, "data", "NewsClipPings", "merged_balanced", "train.json")
METADATA_PATH    = os.path.join(DATASET_ROOT, "data", "NewsClipPings", "metadata", "train.json")
ARTICLE_BASE     = os.path.join(DATASET_ROOT, "origin")  # visual_news/origin/X -> dataset/origin/origin/X

def resolve_article_path(meta_article_path: str) -> str:
    """Convert metadata article_path to actual disk path."""
    rel = meta_article_path.replace("visual_news/", "", 1)
    return os.path.join(ARTICLE_BASE, rel)

def load_article_text(path: str, max_chars: int = 5000) -> str:
    """Load article text (read more to have enough sentences to pick from)."""
    with open(path, "r", encoding="utf-8", errors="replace") as f:
        return f.read(max_chars)

def split_sentences(text: str) -> list[str]:
    """Split text into sentences using regex."""
    sents = re.split(r'(?<=[.!?])\s+', text.strip())
    return [s.strip() for s in sents if len(s.strip()) > 15]

def extract_top_sentences(article_text: str, caption: str, top_k: int = 3) -> str:
    """Extract the top-k most relevant sentences from the article using TF-IDF cosine similarity."""
    sentences = split_sentences(article_text)
    if len(sentences) <= top_k:
        return article_text

    # Build TF-IDF over all sentences + the caption
    corpus = sentences + [caption]
    vectorizer = TfidfVectorizer(stop_words="english")
    tfidf_matrix = vectorizer.fit_transform(corpus)

    # Cosine similarity between each sentence and the caption (last vector)
    caption_vec = tfidf_matrix[-1]
    sent_vecs = tfidf_matrix[:-1]
    sims = cosine_similarity(sent_vecs, caption_vec).flatten()

    # Pick top-k sentence indices, but keep them in original order
    top_indices = sorted(sims.argsort()[-top_k:][::-1])
    selected = [sentences[i] for i in sorted(top_indices)]
    return " ".join(selected)

# Demo
demo_text = "The quick brown fox jumps over the lazy dog. Scientists discovered a new species in the Amazon. The stock market fell sharply today. Weather forecasts predict rain for the weekend. New species found in rainforest exploration."
demo_caption = "A new species was discovered in the Amazon rainforest"
print("Demo extraction:")
print(f"  Caption: {demo_caption}")
print(f"  Extracted: {extract_top_sentences(demo_text, demo_caption, top_k=3)}")
print("\nPaths configured.")

Demo extraction:
  Caption: A new species was discovered in the Amazon rainforest
  Extracted: Scientists discovered a new species in the Amazon. Weather forecasts predict rain for the weekend. New species found in rainforest exploration.

Paths configured.


In [3]:
# ── Load annotations & metadata, sample 100 real + 100 fake ──
with open(ANNOTATIONS_PATH, "r", encoding="utf-8") as f:
    annotations = json.load(f)["annotations"]

with open(METADATA_PATH, "r", encoding="utf-8") as f:
    metadata = json.load(f)

print(f"Annotations: {len(annotations):,}")
print(f"Metadata entries: {len(metadata):,}")

NUM_PER_CLASS = 100
real_samples = []
fake_samples = []

for ann in annotations:
    if len(real_samples) >= NUM_PER_CLASS and len(fake_samples) >= NUM_PER_CLASS:
        break

    art_id = str(ann["id"])
    img_id = str(ann["image_id"])

    if art_id not in metadata or img_id not in metadata:
        continue

    # For NLI we need:
    #   - premise  = the article text for the IMAGE's original article (image_id)
    #   - hypothesis = the caption from the ARTICLE (id)
    # Real pair: same article, so premise supports hypothesis
    # Fake pair: different articles, so premise likely contradicts hypothesis
    
    # Resolve the article text for the image's source article
    img_meta = metadata[img_id]
    art_path = resolve_article_path(img_meta["article_path"])
    if not os.path.isfile(art_path):
        continue

    caption = metadata[art_id]["caption"]
    article_text = load_article_text(art_path)
    if len(article_text.strip()) < 50:
        continue

    entry = {
        "article_id": ann["id"],
        "image_id": ann["image_id"],
        "caption": caption,
        "article_text": article_text,
        "falsified": ann["falsified"],
        "source": metadata[art_id]["source"],
    }

    if not ann["falsified"] and len(real_samples) < NUM_PER_CLASS:
        real_samples.append(entry)
    elif ann["falsified"] and len(fake_samples) < NUM_PER_CLASS:
        fake_samples.append(entry)

samples = real_samples + fake_samples
print(f"\nSelected {len(real_samples)} real + {len(fake_samples)} fake = {len(samples)} samples")

# Preview
s = samples[0]
print(f"\nExample (REAL):")
print(f"  Caption : {s['caption'][:100]}")
print(f"  Article : {s['article_text'][:200]}...")

Annotations: 71,072
Metadata entries: 385,003

Selected 100 real + 100 fake = 200 samples

Example (REAL):
  Caption : Saudi troops cheer as they ride at the back of an army truck in the southwestern province of Jizan n
  Article : Mazrak camp in the tough mountainous scrublands of Yemen's north-west border with Saudi Arabia is now home to more than 10,000 people displaced by the escalating war between the government and rebels ...


In [4]:
# ── Load DeBERTa-v3 NLI model ──
MODEL_NAME = "cross-encoder/nli-deberta-v3-large"

print(f"Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
nli_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME).to(
    torch.device("cuda" if torch.cuda.is_available() else "cpu")
)
nli_model.eval()

device = next(nli_model.parameters()).device
print(f"Model loaded on: {device}")
print(f"Labels: {nli_model.config.id2label}")
param_count = sum(p.numel() for p in nli_model.parameters()) / 1e6
print(f"Parameters: {param_count:.1f}M")

Loading cross-encoder/nli-deberta-v3-large...


Loading weights: 100%|██████████| 394/394 [00:00<00:00, 6031.93it/s]
DebertaV2ForSequenceClassification LOAD REPORT from: cross-encoder/nli-deberta-v3-large
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded on: cuda:0
Labels: {0: 'contradiction', 1: 'entailment', 2: 'neutral'}
Parameters: 435.1M


In [5]:
# ── Run NLI inference with TF-IDF sentence extraction ──
# Instead of feeding the full article (which gets truncated at 512 tokens),
# we extract the 3 most relevant sentences using TF-IDF cosine similarity
# against the caption, then use those as the premise.

label_map = nli_model.config.id2label
results = []

for s in tqdm(samples, desc="NLI inference (TF-IDF extracted)"):
    caption = s["caption"]
    
    # Extract top 3 most relevant sentences from article
    premise = extract_top_sentences(s["article_text"], caption, top_k=3)
    hypothesis = caption

    inputs = tokenizer(
        premise, hypothesis,
        return_tensors="pt",
        truncation=True,
        max_length=512,
        padding=True,
    ).to(device)

    with torch.no_grad():
        logits = nli_model(**inputs).logits
        probs = torch.softmax(logits, dim=1)[0]

    pred_label_id = logits.argmax(dim=1).item()

    results.append({
        "article_id": s["article_id"],
        "image_id": s["image_id"],
        "ground_truth": "REAL" if not s["falsified"] else "FAKE",
        "caption": caption[:60] + "...",
        "extracted_premise": premise[:100] + "...",
        "nli_label": label_map[pred_label_id],
        "prob_contradiction": probs[0].item(),
        "prob_entailment": probs[1].item(),
        "prob_neutral": probs[2].item(),
        "source": s["source"],
    })

print(f"\nDone! Processed {len(results)} samples.")

NLI inference (TF-IDF extracted): 100%|██████████| 200/200 [00:14<00:00, 13.93it/s]


Done! Processed 200 samples.


In [6]:
# ── Evaluation ──
import matplotlib.pyplot as plt

df = pd.DataFrame(results)

# Strategy: predict FAKE if entailment probability is low (below threshold)
# Use entailment prob as the "realness" score
# We'll also try: predict FAKE if contradiction is the top label

# Method 1: Label-based — contradiction or neutral → FAKE, entailment → REAL
df["pred_label"] = df["nli_label"].apply(lambda x: "REAL" if x == "entailment" else "FAKE")

# Method 2: Score-based — use entailment probability with 0.5 threshold
df["pred_score"] = df["prob_entailment"].apply(lambda x: "REAL" if x > 0.5 else "FAKE")

for method, pred_col in [("Label-based (entailment vs rest)", "pred_label"),
                          ("Score-based (entailment prob > 0.5)", "pred_score")]:
    y_true = (df["ground_truth"] == "FAKE").astype(int)
    y_pred = (df[pred_col] == "FAKE").astype(int)

    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec  = recall_score(y_true, y_pred)
    f1   = f1_score(y_true, y_pred)

    print("=" * 60)
    print(f"Method: {method}")
    print("=" * 60)
    print(f"  Accuracy  : {acc:.2%}")
    print(f"  Precision : {prec:.2%}")
    print(f"  Recall    : {rec:.2%}")
    print(f"  F1 Score  : {f1:.2%}")
    print()
    print(classification_report(y_true, y_pred, target_names=["REAL", "FAKE"]))

    cm = confusion_matrix(y_true, y_pred)
    cm_df = pd.DataFrame(cm, index=["Actual REAL", "Actual FAKE"], columns=["Pred REAL", "Pred FAKE"])
    display(cm_df)
    print()

# Score stats
real_ent = df[df["ground_truth"] == "REAL"]["prob_entailment"]
fake_ent = df[df["ground_truth"] == "FAKE"]["prob_entailment"]
print(f"{'─' * 60}")
print(f"REAL — entailment prob: mean={real_ent.mean():.4f}, std={real_ent.std():.4f}")
print(f"FAKE — entailment prob: mean={fake_ent.mean():.4f}, std={fake_ent.std():.4f}")
print(f"Separation: {real_ent.mean() - fake_ent.mean():.4f}")

Method: Label-based (entailment vs rest)
  Accuracy  : 61.50%
  Precision : 57.06%
  Recall    : 93.00%
  F1 Score  : 70.72%

              precision    recall  f1-score   support

        REAL       0.81      0.30      0.44       100
        FAKE       0.57      0.93      0.71       100

    accuracy                           0.61       200
   macro avg       0.69      0.61      0.57       200
weighted avg       0.69      0.61      0.57       200



,Pred REAL,Pred FAKE
Actual REAL,30,70
Actual FAKE,7,93



Method: Score-based (entailment prob > 0.5)
  Accuracy  : 62.50%
  Precision : 57.58%
  Recall    : 95.00%
  F1 Score  : 71.70%

              precision    recall  f1-score   support

        REAL       0.86      0.30      0.44       100
        FAKE       0.58      0.95      0.72       100

    accuracy                           0.62       200
   macro avg       0.72      0.62      0.58       200
weighted avg       0.72      0.62      0.58       200



,Pred REAL,Pred FAKE
Actual REAL,30,70
Actual FAKE,5,95



────────────────────────────────────────────────────────────
REAL — entailment prob: mean=0.3174, std=0.4290
FAKE — entailment prob: mean=0.0778, std=0.2049
Separation: 0.2395


In [ ]:
# ── NLI label distribution ──
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. NLI label counts by ground truth
ax = axes[0]
ct = pd.crosstab(df["ground_truth"], df["nli_label"])
ct.plot(kind="bar", ax=ax, color=["#e74c3c", "#2ecc71", "#3498db"], edgecolor="white")
ax.set_title("NLI Labels by Ground Truth")
ax.set_xlabel("")
ax.set_ylabel("Count")
ax.tick_params(axis="x", rotation=0)
ax.legend(title="NLI Label")

# 2. Entailment probability distribution
ax = axes[1]
ax.hist(real_ent, bins=20, alpha=0.7, color="#2ecc71", label="REAL", edgecolor="white")
ax.hist(fake_ent, bins=20, alpha=0.7, color="#e74c3c", label="FAKE", edgecolor="white")
ax.axvline(x=0.5, color="orange", linestyle="--", linewidth=2, label="Threshold")
ax.set_xlabel("Entailment Probability")
ax.set_ylabel("Count")
ax.set_title("Entailment Prob Distribution")
ax.legend()

# 3. Contradiction probability distribution
ax = axes[2]
real_con = df[df["ground_truth"] == "REAL"]["prob_contradiction"]
fake_con = df[df["ground_truth"] == "FAKE"]["prob_contradiction"]
ax.hist(real_con, bins=20, alpha=0.7, color="#2ecc71", label="REAL", edgecolor="white")
ax.hist(fake_con, bins=20, alpha=0.7, color="#e74c3c", label="FAKE", edgecolor="white")
ax.set_xlabel("Contradiction Probability")
ax.set_ylabel("Count")
ax.set_title("Contradiction Prob Distribution")
ax.legend()

plt.suptitle("DeBERTa-v3 NLI — Text Fact-Checking (200 samples)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("deberta_nli_results.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Misclassified examples ──
df["correct"] = df["ground_truth"] == df["pred_label"]
wrong = df[~df["correct"]]

print(f"Misclassified (label-based): {len(wrong)}/{len(df)}")
print()
if len(wrong) > 0:
    print("Sample misclassifications:")
    for _, row in wrong.head(10).iterrows():
        print(f"  [{row['ground_truth']}→{row['pred_label']}] "
              f"ent={row['prob_entailment']:.3f} con={row['prob_contradiction']:.3f} "
              f"| {row['caption']}")